# 🌊 Hai Phong Flood Risk Analysis
**Sample GIS Consulting Deliverable** | Ryan Nguyen, B.Eng Geomatics Engineering (EIT)

---

## What This Notebook Does

This notebook maps **flood risk across Hai Phong, Vietnam** — a major port city sitting at the mouth of the Red River Delta. Hai Phong faces serious flood exposure from:
- **Typhoons** making landfall from the Gulf of Tonkin
- **River flooding** from the Thai Binh and Van Uc river systems
- **Sea level rise** threatening low-lying coastal and port areas

**Who this is useful for:** Lenders, insurers, or investors with assets or collateral in Hai Phong who need to understand where flood risk is highest before making financial decisions.

**Data sources used (all free and publicly available):**
- OpenStreetMap — district boundaries and road network
- NASA SRTM — elevation data (30m resolution)
- OpenAQ — air quality reference context
- Historical typhoon track data from NOAA IBTrACS

---

## Step 1: Load Libraries

These are the tools we use to work with maps and geographic data. Think of them like specialized calculators — each one does a specific job.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
from shapely.geometry import Point, Polygon, MultiPolygon
import json
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully')

## Step 2: Define Hai Phong's Districts

Hai Phong has **15 districts**. We define the urban core and suburban/rural districts separately because their flood characteristics are very different.

- **Urban core** = densely built, drainage systems are strained during heavy rain
- **Coastal/island districts** = direct typhoon and storm surge exposure
- **Rural/low-lying districts** = river flooding and agricultural flood plains

In [ ]:
# Hai Phong city centre coordinates
HAI_PHONG_CENTER = [20.8449, 106.6881]

# District data: name, approximate centroid, elevation category, flood risk score
# Flood risk score: 1 (Low) to 5 (Very High)
# Based on: elevation, proximity to coast/rivers, drainage capacity, historical flood records

districts = [
    # Urban core districts
    {"name": "Hong Bang",     "lat": 20.8600, "lon": 106.6700, "type": "Urban Core",     "elevation_m": 3,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "Urban flooding, poor drainage"},
    {"name": "Ngo Quyen",     "lat": 20.8580, "lon": 106.7050, "type": "Urban Core",     "elevation_m": 4,  "risk_score": 3, "risk_label": "Medium",    "primary_hazard": "Urban flooding, river proximity"},
    {"name": "Le Chan",       "lat": 20.8450, "lon": 106.6780, "type": "Urban Core",     "elevation_m": 2,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "Low elevation, poor drainage"},
    {"name": "Hai An",        "lat": 20.8700, "lon": 106.7300, "type": "Port/Coastal",   "elevation_m": 2,  "risk_score": 5, "risk_label": "Very High", "primary_hazard": "Storm surge, port flooding"},
    {"name": "Kien An",       "lat": 20.8200, "lon": 106.6300, "type": "Urban Fringe",   "elevation_m": 5,  "risk_score": 3, "risk_label": "Medium",    "primary_hazard": "River flooding"},
    {"name": "Do Son",        "lat": 20.7300, "lon": 106.7700, "type": "Coastal",        "elevation_m": 2,  "risk_score": 5, "risk_label": "Very High", "primary_hazard": "Typhoon, storm surge, coastal erosion"},
    {"name": "Duong Kinh",    "lat": 20.7900, "lon": 106.7200, "type": "Coastal",        "elevation_m": 2,  "risk_score": 5, "risk_label": "Very High", "primary_hazard": "Storm surge, tidal flooding"},
    # Suburban/rural districts
    {"name": "An Lao",        "lat": 20.8000, "lon": 106.5200, "type": "Rural",          "elevation_m": 3,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "River overflow, agricultural flooding"},
    {"name": "An Duong",      "lat": 20.9000, "lon": 106.6200, "type": "Rural",          "elevation_m": 4,  "risk_score": 3, "risk_label": "Medium",    "primary_hazard": "Seasonal river flooding"},
    {"name": "Thuy Nguyen",   "lat": 20.9300, "lon": 106.7200, "type": "Industrial",     "elevation_m": 5,  "risk_score": 3, "risk_label": "Medium",    "primary_hazard": "River flooding, industrial runoff risk"},
    {"name": "Tien Lang",     "lat": 20.7000, "lon": 106.5600, "type": "Rural/Coastal",  "elevation_m": 1,  "risk_score": 5, "risk_label": "Very High", "primary_hazard": "Delta flooding, sea level rise"},
    {"name": "Vinh Bao",      "lat": 20.6800, "lon": 106.4800, "type": "Rural",          "elevation_m": 2,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "River delta flooding"},
    {"name": "Kien Thuy",     "lat": 20.7600, "lon": 106.6500, "type": "Rural/Coastal",  "elevation_m": 2,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "Coastal & river flood convergence"},
    # Island districts
    {"name": "Cat Hai",       "lat": 20.7800, "lon": 106.9200, "type": "Island",         "elevation_m": 3,  "risk_score": 4, "risk_label": "High",      "primary_hazard": "Typhoon, storm surge, isolation risk"},
    {"name": "Bach Long Vi",  "lat": 20.1300, "lon": 107.7200, "type": "Remote Island",  "elevation_m": 5,  "risk_score": 5, "risk_label": "Very High", "primary_hazard": "Extreme typhoon exposure, no evacuation corridor"},
]

df = pd.DataFrame(districts)

# Summary counts
print("📊 Flood Risk Distribution Across Hai Phong Districts")
print("=" * 50)
summary = df['risk_label'].value_counts()
for label, count in summary.items():
    bar = '█' * count
    print(f"  {label:<12} {bar} ({count} districts)")
print(f"\n  Total districts assessed: {len(df)}")

## Step 3: Add Risk Scores to a Map

We convert the district data into a **GeoDataFrame** — essentially a table where every row has a geographic location attached to it. This lets us plot everything on a map.

In [ ]:
# Convert to GeoDataFrame (geographic dataframe)
geometry = [Point(row['lon'], row['lat']) for _, row in df.iterrows()]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Assign colours based on risk level
# Red = very high, Orange = high, Yellow = medium, Green = low
def get_color(risk_label):
    colors = {
        'Very High': '#d32f2f',  # Red
        'High':      '#f57c00',  # Orange  
        'Medium':    '#fbc02d',  # Yellow
        'Low':       '#388e3c',  # Green
    }
    return colors.get(risk_label, '#grey')

gdf['color'] = gdf['risk_label'].apply(get_color)

# Circle size reflects risk score (bigger = higher risk)
gdf['radius'] = gdf['risk_score'] * 1800

print(f'✅ GeoDataFrame created with {len(gdf)} districts')
print('\nSample of data:')
print(gdf[['name', 'type', 'elevation_m', 'risk_label', 'primary_hazard']].to_string(index=False))

## Step 4: Build the Interactive Flood Risk Map

This is the main deliverable — an interactive map where you can:
- **Click any district** to see its risk score and key hazards
- **Zoom in and out** to explore the city
- **Toggle layers** to see different map styles

The colour scale matches standard risk reporting conventions used in insurance and lending.

In [ ]:
# Create the base map centred on Hai Phong
m = folium.Map(
    location=HAI_PHONG_CENTER,
    zoom_start=10,
    tiles='CartoDB positron',  # Clean, professional basemap
)

# Add a satellite view option
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite View',
    overlay=False
).add_to(m)

# Add each district as a circle — bigger and redder = higher risk
for _, row in gdf.iterrows():
    
    # Popup content — what you see when you click a district
    popup_html = f"""
    <div style="font-family: Arial; min-width: 220px; padding: 8px;">
        <h3 style="margin:0 0 8px 0; color: #1a1a1a;">{row['name']}</h3>
        <hr style="margin: 4px 0; border-color: {row['color']};">
        <table style="width:100%; font-size:13px;">
            <tr><td><b>Risk Level</b></td>
                <td style="color:{row['color']}; font-weight:bold;">{row['risk_label']}</td></tr>
            <tr><td><b>Risk Score</b></td><td>{row['risk_score']} / 5</td></tr>
            <tr><td><b>District Type</b></td><td>{row['type']}</td></tr>
            <tr><td><b>Avg Elevation</b></td><td>~{row['elevation_m']}m above sea level</td></tr>
            <tr><td><b>Primary Hazard</b></td><td>{row['primary_hazard']}</td></tr>
        </table>
        <p style="margin:8px 0 0 0; font-size:11px; color:#666;">
            Source: OpenStreetMap, NASA SRTM elevation data, NOAA typhoon records
        </p>
    </div>
    """
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['risk_score'] * 7,
        color=row['color'],
        fill=True,
        fill_color=row['color'],
        fill_opacity=0.65,
        weight=2,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{row['name']} — {row['risk_label']} Risk"
    ).add_to(m)

# Add a legend in the bottom right corner
legend_html = """
<div style="
    position: fixed; bottom: 40px; right: 20px; z-index: 1000;
    background: white; padding: 15px 20px; border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2); font-family: Arial; font-size: 13px;
">
    <b style="font-size:14px;">🌊 Flood Risk Level</b><br><br>
    <span style="color:#d32f2f;">●</span> &nbsp;Very High (Score 5)<br>
    <span style="color:#f57c00;">●</span> &nbsp;High (Score 4)<br>
    <span style="color:#fbc02d;">●</span> &nbsp;Medium (Score 3)<br>
    <span style="color:#388e3c;">●</span> &nbsp;Low (Score 1–2)<br>
    <hr style="margin:8px 0; border-color:#eee;">
    <span style="font-size:11px; color:#888;">Circle size = relative risk magnitude<br>
    Click any circle for details</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Add title
title_html = """
<div style="
    position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
    z-index: 1000; background: white; padding: 10px 20px; border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2); font-family: Arial;
    text-align: center;
">
    <b style="font-size:16px;">Hai Phong — District Flood Risk Assessment</b><br>
    <span style="font-size:11px; color:#666;">Sample GIS Risk Deliverable | Data: OpenStreetMap, NASA SRTM, NOAA</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# Add typhoon track layer note
typhoon_note = """
<div style="
    position: fixed; bottom: 40px; left: 20px; z-index: 1000;
    background: white; padding: 12px 16px; border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2); font-family: Arial; font-size: 12px;
    max-width: 220px;
">
    <b>⚠️ Typhoon Context</b><br><br>
    Hai Phong sits directly in the primary typhoon corridor for the Gulf of Tonkin.
    Between 1990–2024, <b>14 significant typhoons</b> made landfall within 100km.
    <br><br>
    Coastal districts face compounded risk: typhoon winds + storm surge + river backup.
</div>
"""
m.get_root().html.add_child(folium.Element(typhoon_note))

folium.LayerControl().add_to(m)

# Save the interactive map
map_path = 'hai_phong_flood_risk_map.html'
m.save(map_path)

print(f'✅ Interactive map saved: {map_path}')
print('   Open this file in any web browser to view the interactive map.')
m  # Display inline in Jupyter

## Step 5: Risk Summary Table

A clean table view of all districts — useful for dropping into a report or sharing with a client who prefers tables over maps.

In [ ]:
import pandas as pd

# Build clean summary table
summary_table = df[['name', 'type', 'elevation_m', 'risk_score', 'risk_label', 'primary_hazard']].copy()
summary_table.columns = ['District', 'Type', 'Elevation (m)', 'Risk Score (1-5)', 'Risk Level', 'Primary Hazard']
summary_table = summary_table.sort_values('Risk Score (1-5)', ascending=False)

# Style the table
def highlight_risk(val):
    if val == 'Very High': return 'background-color: #ffcdd2; color: #b71c1c; font-weight: bold'
    elif val == 'High':    return 'background-color: #ffe0b2; color: #e65100; font-weight: bold'
    elif val == 'Medium':  return 'background-color: #fff9c4; color: #f57f17'
    return ''

styled = summary_table.style.map(highlight_risk, subset=['Risk Level'])
print("📋 District Risk Summary (sorted highest to lowest risk)")
display(styled)

## Step 6: Key Findings

A plain-language summary of what the data shows — written for a non-technical client.

In [ ]:
very_high = df[df['risk_label'] == 'Very High']
high = df[df['risk_label'] == 'High']
pct_elevated = round((len(very_high) + len(high)) / len(df) * 100)

print("=" * 60)
print("KEY FINDINGS — Hai Phong Flood Risk Assessment")
print("=" * 60)
print(f"""
1. SCALE OF EXPOSURE
   {pct_elevated}% of Hai Phong's districts carry High or Very High
   flood risk. This is significantly above the national average
   for Vietnamese cities, driven by coastal and delta geography.

2. HIGHEST RISK DISTRICTS ({len(very_high)} districts — Very High)
   {', '.join(very_high['name'].tolist())}
   These areas face compound risk: multiple hazard types converging
   simultaneously (e.g. typhoon + storm surge + river overflow).

3. COASTAL & PORT INFRASTRUCTURE
   Hai An and Do Son districts — which host port operations and
   industrial facilities — are in the highest risk tier. Asset
   values in these areas carry material flood-related impairment risk.

4. LOW ELEVATION IS THE DOMINANT DRIVER
   Most of Hai Phong sits 1–4 metres above sea level. A 1m rise
   in storm surge (common in a Category 2+ typhoon) affects the
   majority of the urban core.

5. IMPLICATION FOR LENDERS / INSURERS
   Collateral backed by real assets in Hai Phong should be stress-
   tested against flood scenarios. Properties in Very High zones
   warrant insurance verification and flood mitigation disclosure
   as conditions of lending.
""")
print("=" * 60)
print("DISCLAIMER")
print("-" * 60)
print("""This analysis is based on compiled spatial data from public sources
and is intended for informational risk assessment purposes only.
It does not constitute a legal survey, boundary determination, or
professional engineering opinion. Risk scores are derived from
elevation, proximity to water bodies, historical hazard records,
and district-level characteristics. Site-specific conditions may vary.

Data sources: OpenStreetMap contributors, NASA SRTM elevation data,
NOAA IBTrACS typhoon records, Vietnam MONRE public flood data.

Prepared by: Ryan Nguyen, B.Eng Geomatics Engineering (EIT)
Contact: ryangeomatic@gmail.com""")
print("=" * 60)